In [2]:
import pandas as pd
import numpy as np

# === Datei laden ===
df = pd.read_csv("ParetoFront.csv")  # ggf. Pfad anpassen

# === Zielspalten definieren ===
all_objectives = [
    "Driver Violation",
    "Commute Distance",
    "Transport Machines",
    "Transport Attachments",
    "Machines",
    "Workers",
    "Attachments"
]

# === 1. Paretofront aus Transport Attachments & Attachments extrahieren ===
def pareto_front_2d(points):
    points = np.array(points)
    is_efficient = np.ones(points.shape[0], dtype=bool)
    for i, c in enumerate(points):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(points[is_efficient] < c, axis=1)
                | np.all(points[is_efficient] == c, axis=1)
            )
            is_efficient[i] = True
    return points[is_efficient]

pareto_points = pareto_front_2d(df[["Transport Attachments", "Attachments"]].values)
df_pareto_attach = pd.DataFrame(
    pareto_points, columns=["Transport Attachments", "Attachments"]
).drop_duplicates()

# Solution IDs der Pareto-optimalen Kombinationen ermitteln
pareto_attach_ids = df.merge(df_pareto_attach, on=["Transport Attachments", "Attachments"])[["Solution ID", "Transport Attachments", "Attachments"]]
print("\n🆔 Solution IDs der Pareto-optimalen (Transport Attachments, Attachments):")
print(pareto_attach_ids.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

# === 2. Lösungen erweitern ===
df_expanded = df.drop(columns=["Transport Attachments", "Attachments"]).merge(
    pareto_attach_ids, how="cross"
)
# Kombiniere Solution ID_x und Solution ID_y zu einer neuen Spalte "Solution ID"
df_expanded["Solution ID"] = df_expanded["Solution ID_x"].astype(str) + "_" + df_expanded["Solution ID_y"].astype(str)

# Die neue "Solution ID" Spalte an den Anfang verschieben und die alten entfernen
cols = ["Solution ID"] + [col for col in df_expanded.columns if col not in ["Solution ID", "Solution ID_x", "Solution ID_y", "Solution ID_X_Y"]]
df_expanded = df_expanded[cols]

# === 3. Finaler Paretofilter (alle Ziele) ===
def pareto_filter_nd(df, objective_cols):
    values = df[objective_cols].values
    is_efficient = np.ones(values.shape[0], dtype=bool)
    for i, v in enumerate(values):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(values[is_efficient] < v, axis=1)
                | np.all(values[is_efficient] == v, axis=1)
            )
            is_efficient[i] = True
    return df[is_efficient].reset_index(drop=True)

df_final_pareto = pareto_filter_nd(df_expanded, all_objectives)

# Sortiere die Spalten in der gewünschten Reihenfolge
sort_cols = [
    "Solution ID",
    "Orders",
    "Order Items",
    "Driver Violation",
    "Commute Distance",
    "Transport Machines",
    "Transport Attachments",
    "Machines",
    "Workers",
    "Attachments"
]
df_final_pareto = df_final_pareto[sort_cols]

# === Insights ===
print("-"*40)
print("🔍 Insights zur Paretoanalyse\n" + "-"*40)
print(f"📦 Ursprüngliche Lösungen:         {len(df)}")
print(f"🎯 Pareto-Kombinationen (2D):      {len(df_pareto_attach)}")
print(f"🧩 Erweiterte Lösungskombis:       {len(df_expanded)}")
print(f"✅ Nicht-dominierte Endlösungen:   {len(df_final_pareto)}")
print(f"❌ Entfernte (dominierte) Lösungen: {len(df_expanded) - len(df_final_pareto)}\n")

print("📊 Pareto-Kombinationen (Anbaugeräte):")
print(df_pareto_attach.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

# === Ergebnis speichern ===
df_final_pareto.to_csv("ParetoFront_filtered.csv", index=False)
print(f"\n💾 Datei gespeichert unter: ParetoFront_filtered.csv")



🆔 Solution IDs der Pareto-optimalen (Transport Attachments, Attachments):
 Solution ID  Transport Attachments  Attachments
         210                1462.17           57
         178                4932.71           44
----------------------------------------
🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:         979
🎯 Pareto-Kombinationen (2D):      2
🧩 Erweiterte Lösungskombis:       1958
✅ Nicht-dominierte Endlösungen:   38
❌ Entfernte (dominierte) Lösungen: 1920

📊 Pareto-Kombinationen (Anbaugeräte):
 Transport Attachments  Attachments
               1462.17         57.0
               4932.71         44.0

💾 Datei gespeichert unter: ParetoFront_filtered.csv


In [3]:
# Inhalt der gefilterten Paretofront ausgeben
print("\n📄 Inhalt der gefilterten Paretofront:")
print("-" * 40)
print(df_final_pareto.to_string(index=False))
print("-" * 40)
print("Lösungen:", len(df_final_pareto))



📄 Inhalt der gefilterten Paretofront:
----------------------------------------
Solution ID  Orders  Order Items  Driver Violation  Commute Distance  Transport Machines  Transport Attachments  Machines  Workers  Attachments
      1_178      85          884               365         323458.00            64794.07                4932.71        65      118           44
      1_210      85          884               365         323458.00            64794.07                1462.17        65      118           57
      2_178      85          884               366         322743.89            64829.81                4932.71        65      118           44
      2_210      85          884               366         322743.89            64829.81                1462.17        65      118           57
      3_178      85          884               367         322760.73            64814.37                4932.71        65      118           44
      3_210      85          884               367      

In [2]:
import json
import pandas as pd
from pathlib import Path

# === Pfade definieren ===
solutions_path = Path("pareto_solutions.json")
filtered_path = Path("ParetoFront_filtered.csv")
output_path = Path("pareto_solutions_filtered.json")

# === Dateien laden ===

# 1. JSON mit vollständigen Lösungen
with open(solutions_path, "r", encoding="utf-8") as f:
    solutions = json.load(f)

# 2. CSV mit kombinierten IDs (z. B. "1_178")
df_filtered = pd.read_csv(filtered_path)

# Sicherstellen, dass die relevante Spalte existiert
if "Solution ID" not in df_filtered.columns:
    raise ValueError("Die CSV-Datei muss eine Spalte 'Solution ID' enthalten.")

# === Neue kombinierte Lösungen erstellen ===
combined = {}

for _, row in df_filtered.iterrows():
    combined_id = row["Solution ID"]
    base_id, attach_id = combined_id.split("_")

    if base_id not in solutions or attach_id not in solutions:
        print(f"⚠️ ID {combined_id} enthält ungültige Referenz: {base_id} oder {attach_id} nicht gefunden.")
        continue

    base = solutions[base_id]
    attach = solutions[attach_id]

    combined[combined_id] = {
        "worker_route_plan": base["worker_route_plan"],
        "machine_route_plan": base["machine_route_plan"],
        "attachment_route_plan": attach["attachment_route_plan"],
        "combined_from": {
            "worker_machine_id": base_id,
            "attachment_id": attach_id
        }
    }

# === Neue Datei speichern ===
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

print(f"✅ {len(combined)} kombinierte Lösungen gespeichert unter: {output_path}")

✅ 38 kombinierte Lösungen gespeichert unter: pareto_solutions_filtered.json
